[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/stevenelliottjr/healthcare-analytics-course-data/blob/main/notebooks/lab06_evaluation_subgroups.ipynb) · [Course home](https://colab.research.google.com/github/stevenelliottjr/healthcare-analytics-course-data/blob/main/notebooks/00_course_home.ipynb)

**Google Colab version.** **Before you start: File → Save a copy in Drive.** Work only in your copy (its title starts with *Copy of*); changes to this original are not saved. Rename your copy `Lab 6 — <your last name>` (click the title at the top left).

**To submit:** **Runtime → Run all** and check that every cell ran, then **File → Save**. Click
**Share** (top right), add **steven.elliott@student.fairfield.edu** as **Editor**, and send. Colab keeps your saved versions
under **File → Revision history**; **File → Save and pin revision** keeps one permanently, which is how you
mark a submission.

**Coming back later?** Colab forgets variables and downloaded files when the runtime disconnects (after about 90 minutes idle). Open your copy and click **Runtime → Run all**.

# Lab 6 — Predicting outcomes II: evaluation and subgroups
*Week 6. Baseline: sections 1–5. Powerup: section 6.*

Last week you built a model that predicts which hospitals land in Medicare's heart-failure readmission
penalty range. This week you find out how good it really is. One number (accuracy, or even AUROC) hides
almost everything a decision-maker needs: how well the model **ranks**, whether its probabilities can be
**taken at face value**, and whether it works equally well for **every kind of hospital**. Sometimes the
subgroup gap is the most important finding in the whole analysis.

**How labs work.** Save a copy in Drive, work in your copy, share it back with the instructor
when you submit. Every lab has a **baseline** (everyone finishes it in the session) and
**powerups** (optional depth). AI assistants may write code for you; **you own every line** and
may be asked to explain any of it. Cells marked `# TODO` are yours to fill in.

In [ ]:
# --- Course setup: run this cell first -------------------------------------------
# Loaders for the public datasets this course uses. The canonical copy of this code
# is labs/course_data.py in the course repo; you will read it line by line in week 3.
import io, json, urllib.request, urllib.parse
from pathlib import Path
import pandas as pd
import duckdb

CACHE = Path("data"); CACHE.mkdir(exist_ok=True)

def sql(query: str) -> pd.DataFrame:
    """Run SQL over any DataFrame in this notebook by its variable name (DuckDB)."""
    return duckdb.sql(query).df()

def _get(url, params=None, timeout=120) -> bytes:
    if params:
        url = url + "?" + urllib.parse.urlencode(params)
    req = urllib.request.Request(url, headers={"User-Agent": "healthcare-analytics-course/1.0"})
    with urllib.request.urlopen(req, timeout=timeout) as r:
        return r.read()

# ---- CMS Care Compare (data.cms.gov/provider-data) --------------------------------
# File names change at every CMS refresh; dataset identifiers do not. We ask the
# catalog for the current CSV URL every time instead of hard-coding a file name.
CMS_CATALOG = "https://data.cms.gov/provider-data/api/1/metastore/schemas/dataset/items"
CMS_IDS = {
    "hospital_general_info": "xubh-q36u",   # one row per hospital
    "hrrp":                  "9n3s-kdb3",   # Hospital Readmissions Reduction Program, one row per hospital x measure
    "unplanned_visits":      "632h-zaca",   # readmissions / ED visits, one row per hospital x measure
    "complications_deaths":  "ynj2-r877",
    "timely_effective":      "yv7e-xc69",
}

def cms_csv_url(identifier: str) -> str:
    # Ask for the one dataset by id; the full keyword search can take minutes or time out.
    item = json.loads(_get(f"{CMS_CATALOG}/{identifier}", timeout=30))
    for d in item.get("distribution", []):
        if d.get("mediaType") == "text/csv":
            return d["downloadURL"]
    raise KeyError(f"No CSV distribution found for CMS dataset {identifier}")

# Course mirror: a snapshot of every CMS file the labs use, for when CMS is down or rate-limits
# a classroom downloading the same file at once. The live source is always tried first.
COURSE_DATA_BASE = "https://raw.githubusercontent.com/stevenelliottjr/healthcare-analytics-course-data/main/data"

def _fetch_or_mirror(path: Path, fetch, mirror_file: str) -> None:
    """Save `path` from the live source; if that fails, from the course mirror."""
    try:
        path.write_bytes(fetch())
    except Exception as e:
        print(f"Live download failed ({type(e).__name__}); using the course mirror for {path.name}.")
        path.write_bytes(_get(f"{COURSE_DATA_BASE}/{mirror_file}", timeout=300))

def load_cms(name: str, **read_csv_kwargs) -> pd.DataFrame:
    """Download (once) and read a Care Compare file. Everything is read as TEXT on purpose:
    IDs keep their leading zeros and 'Not Available' stays visible instead of silently
    becoming NaN. Converting columns to numbers is your job, and part of the lesson."""
    path = CACHE / f"cms_{name}.csv"
    if not path.exists():
        _fetch_or_mirror(path, lambda: _get(cms_csv_url(CMS_IDS[name]), timeout=300), f"cms/cms_{name}.csv")
    kw = dict(dtype=str, keep_default_na=False, encoding="utf-8-sig")
    kw.update(read_csv_kwargs)
    return pd.read_csv(path, **kw)

def to_num(s: pd.Series) -> pd.Series:
    """Text -> number; anything that is not a number ('Not Available', 'N/A', '') becomes NaN."""
    return pd.to_numeric(s, errors="coerce")

# ---- CDC PLACES, county release (Socrata API on data.cdc.gov) ---------------------
PLACES_COUNTY = "https://data.cdc.gov/resource/swc5-untb.json"
PLACES_SELECT = ("year,stateabbr,statedesc,locationname,locationid,categoryid,measureid,measure,"
                 "data_value,low_confidence_limit,high_confidence_limit,totalpopulation")

def load_places(measure_ids, data_value_type="Age-adjusted prevalence", state=None, page=50000) -> pd.DataFrame:
    """County-level model-based prevalence estimates (percent of adults). locationid is the
    5-digit county FIPS. Note: Connecticut reports *planning regions*, not counties, since 2022."""
    where = "measureid in(" + ",".join(f"'{m}'" for m in measure_ids) + f") AND data_value_type='{data_value_type}'"
    if state:
        where += f" AND stateabbr='{state}'"
    frames, offset = [], 0
    while True:
        chunk = pd.DataFrame(json.loads(_get(PLACES_COUNTY, {"$where": where, "$select": PLACES_SELECT,
                                                               "$limit": page, "$offset": offset})))
        frames.append(chunk)
        if len(chunk) < page:
            break
        offset += page
    df = pd.concat(frames, ignore_index=True)
    for c in ["data_value", "low_confidence_limit", "high_confidence_limit", "totalpopulation"]:
        df[c] = pd.to_numeric(df[c], errors="coerce")
    return df

# ---- Week 4 powerups (links checked 2026-10-26; 2022 is the latest SVI release) ----
SVI_COUNTY_URL = "https://svi.cdc.gov/Documents/Data/2022/csv/states_counties/SVI_2022_US_county.csv"
def load_svi_county() -> pd.DataFrame:
    """CDC/ATSDR Social Vulnerability Index 2022, one row per county.
    Key columns: FIPS, STATE, COUNTY, E_TOTPOP, RPL_THEMES (overall percentile 0-1),
    RPL_THEME1..RPL_THEME4. The value -999 means 'not available'."""
    path = CACHE / "svi_2022_county.csv"
    if not path.exists():
        path.write_bytes(_get(SVI_COUNTY_URL, timeout=300))
    return pd.read_csv(path, dtype={"FIPS": str}).replace(-999, pd.NA)

CHR_URL = "https://www.countyhealthrankings.org/sites/default/files/media/document/analytic_data2025_v3.csv"
def load_county_health_rankings() -> pd.DataFrame:
    """County Health Rankings analytic file. Row 0 holds readable labels and row 1 the variable
    codes; we skip the labels so the codes (fipscode, v001..v1xx) become the column names.
    The data dictionary on the CHR site maps each v-code to a measure."""
    path = CACHE / "chr_analytic_2025_v3.csv"
    if not path.exists():
        path.write_bytes(_get(CHR_URL, timeout=300))
    return pd.read_csv(path, skiprows=[0], dtype={"fipscode": str, "statecode": str, "countycode": str})

print("course setup ready")


In [ ]:
# --- Week 6 packages: scikit-learn -------------------------------------------------
# Not every base image has it (Hex does not). Installs once per kernel session.
import importlib.util, shutil, subprocess, sys

_need = [pkg for pkg, mod in [("scikit-learn", "sklearn")] if importlib.util.find_spec(mod) is None]
if _need:
    if shutil.which("uv"):  # Hex: the kernel's virtualenv has no pip module, but uv is on the PATH
        subprocess.run(["uv", "pip", "install", "-q", "--python", sys.executable, *_need], check=True)
    else:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *_need], check=True)
    importlib.invalidate_caches()
print("week 6 packages ready")


## 1. The model you are evaluating (given)
So everyone evaluates the same model, the next cell rebuilds a clean version of the lab 5 baseline: target
`penalized` (heart-failure Excess Readmission Ratio > 1), the predictors that survived the leakage check
(ownership, emergency services, state, and log of heart-failure discharges), a logistic regression, and the
same stratified 80/20 split with `random_state=42`. Your own lab 5 model may differ a little. That is fine.

It leaves you a dataframe `test`, one row per held-out hospital, with the true label `penalized` and the
model's predicted probability `p_hat`. Everything this week is computed from those two columns.

**Baseline:** report the size of the test set and its penalty rate. Then look at `unscored`, the hospitals
the model *cannot* score because CMS did not publish their number of discharges. How many are there, and
what is their penalty rate compared with the hospitals the model does score?

In [ ]:
# --- Given: the lab 5 baseline model, rebuilt the same way for everyone ------------
import numpy as np
import matplotlib.pyplot as plt
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

hrrp = load_cms("hrrp")
hf = hrrp[hrrp["Measure Name"].eq("READM-30-HF-HRRP")].copy()
hf["excess_ratio"] = to_num(hf["Excess Readmission Ratio"])
hf = hf.dropna(subset=["excess_ratio"])
hf["penalized"] = (hf["excess_ratio"] > 1).astype(int)

hgi = load_cms("hospital_general_info")
df = hf[["Facility ID", "penalized", "Number of Discharges"]].merge(hgi, on="Facility ID", how="inner")
m = df.rename(columns=lambda c: c.strip().lower().replace(" ", "_").replace("/", "_"))
m["number_of_discharges"] = to_num(m["number_of_discharges"])
unscored = m[m["number_of_discharges"].isna()]           # no discharge count, so the model cannot score them
m = m.dropna(subset=["number_of_discharges"]).reset_index(drop=True)
m["log_discharges"] = np.log(m["number_of_discharges"])

NUMERIC = ["log_discharges"]
CATEGORICAL = ["hospital_ownership", "emergency_services", "state"]
FEATURES = NUMERIC + CATEGORICAL

train, test = train_test_split(m, test_size=0.2, stratify=m["penalized"], random_state=42)

def make_model(clf):
    return Pipeline([
        ("prep", ColumnTransformer([("num", StandardScaler(), NUMERIC),
                                    ("cat", OneHotEncoder(handle_unknown="ignore"), CATEGORICAL)])),
        ("clf", clf),
    ])

model = make_model(LogisticRegression(max_iter=1000)).fit(train[FEATURES], train["penalized"])

test = test.copy()
test["p_hat"] = model.predict_proba(test[FEATURES])[:, 1]   # predicted probability of being penalized
y, p = test["penalized"], test["p_hat"]
print(f"scored: train {len(train)}, test {len(test)}; unscored: {len(unscored)}")


In [ ]:
# TODO: test-set size and penalty rate; number of unscored hospitals and their penalty rate.


**Who is in the test set, and who is missing?** *(edit this cell)*

## 2. Ranking: the ROC curve and AUROC
The ROC curve sweeps every possible threshold on `p_hat` and plots the share of penalized hospitals caught
(true positive rate, *sensitivity*) against the share of non-penalized hospitals wrongly flagged (false
positive rate). **AUROC** is the area under it: the probability that a randomly chosen penalized hospital gets
a higher `p_hat` than a randomly chosen non-penalized one. 0.5 is a coin flip; 1.0 is perfect ranking.

**Baseline:** draw the ROC curve, report the AUROC, and write one sentence that explains the number to a
hospital CFO without using the words "area" or "curve".

In [ ]:
from sklearn.metrics import roc_auc_score, roc_curve

# TODO: fpr, tpr, thresholds = roc_curve(y, p);  auroc = roc_auc_score(y, p)
fig_roc, ax_roc = plt.subplots(figsize=(4.5, 4.5))
# TODO: ax_roc.plot(fpr, tpr, label=f"logistic (AUROC {auroc:.3f})")
ax_roc.plot([0, 1], [0, 1], "--", color="grey", label="coin flip (0.5)")
ax_roc.set_xlabel("False positive rate (1 - specificity)")
ax_roc.set_ylabel("True positive rate (sensitivity)")
ax_roc.set_title("ROC, hold-out hospitals")
ax_roc.legend(loc="lower right"); plt.show()


**AUROC for a CFO, in one sentence:** *(edit this cell)*

## 3. Precision, recall, and AUPRC
Suppose a state hospital association uses the model to pick hospitals for a readmission-reduction program.
What it cares about is **precision** (of the hospitals we flag, how many really are in the penalty range) and
**recall** (of the hospitals in the penalty range, how many we flag). The precision-recall curve plots one
against the other across thresholds; **AUPRC** (average precision) summarizes it.

The floor for AUPRC is not 0.5. A model that guesses at random has precision equal to the **base rate**, so
that is the number to compare against.

**Baseline:**
1. Draw the precision-recall curve with a horizontal line at the test-set base rate. Report AUPRC and the base rate.
2. If the program can only take the **top 20%** of hospitals by `p_hat`, what share of them are actually
   penalized? Compare that with the base rate.
3. One or two sentences: AUROC and AUPRC tell a similar story here. Why? For which outcome in this course
   would AUPRC be the number to watch?

In [ ]:
from sklearn.metrics import average_precision_score, precision_recall_curve

base_rate = y.mean()
# TODO: precision, recall, _ = precision_recall_curve(y, p);  auprc = average_precision_score(y, p)
fig_pr, ax_pr = plt.subplots(figsize=(4.5, 4.5))
# TODO: ax_pr.plot(recall, precision, label=f"logistic (AUPRC {auprc:.3f})")
ax_pr.axhline(base_rate, ls="--", color="grey", label=f"base rate ({base_rate:.3f})")
ax_pr.set_xlabel("Recall"); ax_pr.set_ylabel("Precision"); ax_pr.set_ylim(0, 1.02)
ax_pr.set_title("Precision-recall, hold-out hospitals")
ax_pr.legend(loc="lower left"); plt.show()

# TODO: precision in the top 20%:  test.nlargest(int(0.2 * len(test)), "p_hat")["penalized"].mean()


**Top 20%, and why AUROC and AUPRC agree here:** *(edit this cell)*

## 4. Calibration: can you take the probabilities at face value?
A model can rank well and still be wrong about the size of the risk. **Calibration** asks: of the hospitals
the model gives about a 70% chance, are about 70% actually penalized? A **reliability curve** bins hospitals
by `p_hat` (here, 10 bins with equal numbers of hospitals) and plots the mean prediction in each bin against
the observed rate. A perfectly calibrated model sits on the diagonal.

The **Brier score** is the mean squared error of the probabilities (lower is better). Its floor is the Brier
score of predicting the training-set base rate for every hospital, a model that knows nothing.

**Baseline:** draw the reliability curve. Report the Brier score next to the know-nothing Brier score. Two
sentences: where is the model over- or under-confident, and does the Brier improvement look large or small
to you?

In [ ]:
from sklearn.calibration import calibration_curve
from sklearn.metrics import brier_score_loss

# TODO: observed, predicted = calibration_curve(y, p, n_bins=10, strategy="quantile")
fig_cal, ax_cal = plt.subplots(figsize=(4.5, 4.5))
# TODO: ax_cal.plot(predicted, observed, "o-", label="logistic")
ax_cal.plot([0, 1], [0, 1], "--", color="grey", label="perfectly calibrated")
ax_cal.set_xlabel("Mean predicted probability (bin)"); ax_cal.set_ylabel("Observed penalty rate (bin)")
ax_cal.set_title("Reliability curve, 10 equal-size bins")
ax_cal.legend(loc="upper left"); plt.show()

# TODO: brier = brier_score_loss(y, p)
# TODO: brier_floor = brier_score_loss(y, np.full(len(y), train["penalized"].mean()))


**Where is it miscalibrated, and is the Brier gain large?** *(edit this cell)*

## 5. Subgroups: does it work for every kind of hospital?
An overall AUROC is an average over very different hospitals. The cell below adds a few ways to slice the
test set: `ownership_group` (non-profit, for-profit, government), `size` (thirds of heart-failure
discharges), `region` (Census region), `star_rating` (rated or not), `emergency_services`, and
`hospital_type`. Rural/urban status is not in these CMS files; you would need an outside source such as the
USDA's rural-urban codes by ZIP, which makes a good capstone extension.

**Baseline:**
1. Finish `subgroup_metrics(d)` so it returns, for a slice `d` of the test set: `n`, `observed` (penalty
   rate), `predicted` (mean `p_hat`), `auroc`, and `brier`. AUROC needs both classes present; return `np.nan`
   when it is not. Run the loop to get the table.
2. Why does `hospital_type` have only one row?
3. Pick **one** subgroup where the model underperforms, either a low AUROC or a gap between `observed` and
   `predicted`. Say which, with numbers. Then argue whether it is a **fairness finding** (the model is
   systematically worse for a group, in a way that would matter if someone acted on it) or an **artifact**
   (too few hospitals to tell, or a slice that removes the variation the model relies on). What evidence
   would settle it?

*If you finish early:* resample the subgroup's rows 1,000 times with replacement, recompute AUROC each time,
and report the middle 95% of those values as a bootstrap confidence interval.

In [ ]:
# --- Given: subgroup columns for the test set ----------------------------------------
def ownership_group_of(s):
    if s.startswith("Voluntary"):
        return "Non-profit"
    if s.startswith("Government"):
        return "Government"
    if s in ("Proprietary", "Physician"):
        return "For-profit"
    return "Other"   # Tribal, and anything new CMS adds

REGIONS = {
    "Northeast": "CT ME MA NH RI VT NJ NY PA",
    "Midwest":   "IL IN MI OH WI IA KS MN MO NE ND SD",
    "South":     "DE DC FL GA MD NC SC VA WV AL KY MS TN AR LA OK TX",
    "West":      "AZ CO ID MT NV NM UT WY AK CA HI OR WA",
}
STATE_TO_REGION = {st: region for region, states in REGIONS.items() for st in states.split()}

test["ownership_group"] = test["hospital_ownership"].map(ownership_group_of)
test["size"] = pd.qcut(test["number_of_discharges"], 3, labels=["small", "medium", "large"])
test["region"] = test["state"].map(STATE_TO_REGION).fillna("Territory")
test["star_rating"] = np.where(test["hospital_overall_rating"].eq("Not Available"), "not rated", "rated")

SUBGROUPS = ["hospital_type", "ownership_group", "size", "region", "star_rating", "emergency_services"]
test[SUBGROUPS].describe()


In [ ]:
def subgroup_metrics(d):
    # TODO: return a dict with n, observed, predicted, auroc, brier for the rows in d.
    # Hint: roc_auc_score(d["penalized"], d["p_hat"]) if d["penalized"].nunique() == 2 else np.nan
    return {}

rows = []
for col in SUBGROUPS:
    for level, d in test.groupby(col, observed=True):
        rows.append({"subgroup": col, "level": level, **subgroup_metrics(d)})
sub = pd.DataFrame(rows)
sub


**One row for hospital_type; the subgroup I picked; fairness finding or artifact?** *(edit this cell)*

## 6. Powerup: track three model configurations in MLflow
Once you try more than two models, you lose track of which settings produced which numbers. **MLflow** records
every run: the parameters you chose, the metrics you got, and when.

Run the packages cell below, then log **three configurations** (for example the baseline logistic regression,
a more strongly regularized one, `LogisticRegression(C=0.05, max_iter=1000)`, and a
`DecisionTreeClassifier(max_depth=4, min_samples_leaf=20, random_state=42)`) using `make_model(...)` and the
same split. For each run, log the parameters and the hold-out AUROC, AUPRC, and Brier score. Then pull all the
runs back as a dataframe with `mlflow.search_runs()` and say which configuration you would carry forward, and why.

MLflow needs somewhere to write. In Colab there is no tracking server, so use a local SQLite file:
`mlflow.set_tracking_uri("sqlite:///mlruns.db")`. (MLflow 3 no longer accepts the old `file:./mlruns` folder
store. The file disappears when the Colab runtime disconnects; that is fine for this exercise. To keep runs between sessions, mount Drive (Files panel → Drive icon) and use `sqlite:////content/drive/MyDrive/mlruns.db`.)

In [ ]:
# --- Powerup packages: MLflow ----------------------------------------------------------
# Same pattern as the week 6 packages cell, wrapped in a function so no names are redefined.
# sqlalchemy and alembic are what MLflow needs for its SQLite tracking store.
def _install_mlflow():
    import importlib, importlib.util, shutil, subprocess, sys
    if importlib.util.find_spec("mlflow") is None:
        if shutil.which("uv"):
            cmd = ["uv", "pip", "install", "-q", "--python", sys.executable]
        else:
            cmd = [sys.executable, "-m", "pip", "install", "-q"]
        subprocess.run(cmd + ["mlflow-skinny", "sqlalchemy", "alembic"], check=True)
        importlib.invalidate_caches()
    # Hex loads an older OpenTelemetry when the kernel starts; MLflow needs the copy it just installed.
    for name in [m for m in sys.modules if m.split(".")[0] == "opentelemetry"]:
        del sys.modules[name]

_install_mlflow()
print("mlflow ready")


In [ ]:
import mlflow
from sklearn.tree import DecisionTreeClassifier

mlflow.set_tracking_uri("sqlite:///mlruns.db")
mlflow.set_experiment("lab06-hf-penalty")

CONFIGS = {
    "logistic_baseline": LogisticRegression(max_iter=1000),
    # TODO: two more configurations
}
# TODO: for name, clf in CONFIGS.items():
#           with mlflow.start_run(run_name=name):
#               fit make_model(clf) on train, predict on test,
#               mlflow.log_params(clf.get_params()) (or just the ones you changed),
#               mlflow.log_metrics({"auroc": ..., "auprc": ..., "brier": ...})
# runs = mlflow.search_runs(); runs[["tags.mlflow.runName", "metrics.auroc", "metrics.auprc", "metrics.brier"]]


**Which configuration would you carry forward, and why?** *(edit this cell)*

## Milestone — analysis plan (due with this lab)
One page, in the cell below, for your capstone question. A plan is what you commit to *before* you see the
results, so you cannot quietly move the goalposts later. Use this week's vocabulary: name the metric, the
floor it has to beat, and the subgroups you will check.

**Question (from your week 2 brief, revised if it changed):**

**Outcome:** what exactly, how it is measured, and its base rate or distribution.

**Unit of analysis and population:** one row is a … ; who is included and who is excluded (and why).

**Predictors:** each one, its source, and why it does not leak.

**Method:** the model or comparison, and the simple floor it must beat.

**How success is judged:** the metric(s), the number that would count as useful, and the subgroups you will
check.

**What would change the plan:**